<a href="https://colab.research.google.com/github/saadbinather/GNN-based-LLM-Halucination-Detection/blob/main/wikidata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install spacy sparqlwrapper
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 11.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 89.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
import spacy
from SPARQLWrapper import SPARQLWrapper, JSON

# 1. NER Setup
nlp = spacy.load("en_core_web_sm")
text = "Apple and Microsoft are tech giants based in USA."
doc = nlp(text)

# 2. Wikidata SPARQL Setup
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

def get_wikidata_features(entity_name):
    # Simple query: Entity label se match karke description aur website nikalna
    query = f"""
    SELECT ?itemLabel ?description ?website WHERE {{
      ?item rdfs:label "{entity_name}"@en.
      OPTIONAL {{ ?item schema:description ?description. FILTER(LANG(?description) = "en") }}
      OPTIONAL {{ ?item wdt:P856 ?website. }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }} LIMIT 1
    """
    sparql.setQuery(query)
    try:
        results = sparql.query().convert()
        return results["results"]["bindings"]
    except:
        return []

# 3. Execution
print(f"Text: {text}\n" + "-"*30)

for ent in doc.ents:
    if ent.label_ in ["ORG", "GPE", "PERSON"]:  # Sirf relevant entities uthayein
        print(f"Entity: {ent.text} ({ent.label_})")
        features = get_wikidata_features(ent.text)

        if features:
            desc = features[0].get("description", {}).get("value", "No description found")
            site = features[0].get("website", {}).get("value", "No website found")
            print(f" - KG Description: {desc}")
            print(f" - Official Website: {site}")
        else:
            print(" - No Wikidata features found.")
        print()

Text: Apple and Microsoft are tech giants based in USA.
------------------------------
Entity: Apple (ORG)
 - KG Description: Wikimedia disambiguation page
 - Official Website: No website found

Entity: Microsoft (ORG)
 - KG Description: American multinational technology corporation
 - Official Website: https://www.microsoft.com/

Entity: USA (GPE)
 - KG Description: Wikimedia disambiguation page
 - Official Website: No website found

